# 42. Numbers in the paper

Computes the key numbers quoted in `NewGlenn_explosion_paper.tex` from the CSV products of notebooks 21–38 and
writes them as LaTeX macros to `tables/numbers.tex` (read by the paper with `\input{tables/numbers}`). Rerun
after notebooks 21–40, then recompile the paper. Each macro is printed with its value for checking.

In [1]:
import numpy as np
import pandas as pd
import newglenn_config as cfg

O = cfg.OUT_DIR
V = {}

def rng(a, b, d=0, sep='--'):
    return f'{a:.{d}f}{sep}{b:.{d}f}'

## Origin time and weather (notebook 24)

In [2]:
fit = pd.read_csv(O / '24_origin_time' / 'origin_time_fit.csv')
best = fit[(fit.onset == 'onset_05') & (fit.model == 'weather + wind')].iloc[0]
t = pd.to_datetime(best.origin_time)
V['NGorigin'] = (t.round('100ms')).strftime('%H:%M:%S.%f')[:10]
V['NGoriginPrecise'] = (t.round('10ms')).strftime('%H:%M:%S.%f')[:11]
V['NGoriginSpread'] = f'{best.spread_s:.2f}'
wb = fit[fit.model != 'free c']
tt = pd.to_datetime(wb.origin_time)
V['NGoriginRange'] = tt.min().round('100ms').strftime('%H:%M:%S.%f')[:10] + '--' + tt.max().round('100ms').strftime('%S.%f')[:4]
free = fit[(fit.onset == 'onset_05') & (fit.model == 'free c')].iloc[0]
V['NGoriginFree'] = pd.to_datetime(free.origin_time).round('100ms').strftime('%H:%M:%S.%f')[:10]
V['NGspeedFree'] = f'{free.c_mps:.0f}'
w = pd.read_csv(O / '24_origin_time' / 'weather_at_origin.csv').set_index('station')
m = w.loc[cfg.EVENT.raw['weather']['stations']].mean()
from ksclib import weather
spd, frm = weather.wind_from_uv(m.u, m.v)
V['NGtemp'] = f'{m.temperature_c:.1f}'
V['NGrh'] = f'{m.rh:.0f}'
V['NGpressure'] = f'{m.pressure_pa / 100:.0f}'
V['NGwind'] = f'{float(spd):.1f}'
V['NGwindFrom'] = f'{float(frm):.0f}'
sp = pd.read_csv(O / '24_origin_time' / 'airwave_speeds.csv').set_index('station').loc[['B20', 'B14', 'B29', 'B05']]
V['NGczero'] = f'{sp.c0_mps.iloc[0]:.1f}'
V['NGceff'] = rng(sp.c_eff_mps.min(), sp.c_eff_mps.max(), 1)
V['NGheadwind'] = rng(-sp.wind_along_ray_mps.max(), -sp.wind_along_ray_mps.min(), 1)

## Blast wave, decay and yield (notebook 34)

In [3]:
bp = pd.read_csv(O / '34_amplitude_vs_distance' / 'blast_parameters.csv')
b20 = bp[bp.station == 'B20'].iloc[0]
far = bp[bp.station != 'B20']
V['NGpBtwenty'] = f'{b20.P_pos_Pa:.0f}'
V['NGpFar'] = rng(far.P_pos_Pa.min(), far.P_pos_Pa.max())
V['NGtpos'] = rng(bp.t_pos_s.min(), bp.t_pos_s.max(), 2)
V['NGipos'] = rng(bp.I_pos_Pa_s.min(), bp.I_pos_Pa_s.max())
V['NGiposr'] = rng(np.round(bp.I_pos_x_r_Pa_s_km.min(), -1), np.round(bp.I_pos_x_r_Pa_s_km.max(), -1))
ratio = bp.P_pos_Pa / bp.P_neg_Pa.abs()
V['NGnegRatio'] = rng(np.round(ratio.min()), np.round(ratio.max()))
dec = pd.read_csv(O / '34_amplitude_vs_distance' / 'decay_fits.csv')
def n(metric, data='all', d=2):
    r = dec[(dec.metric == metric) & (dec.data == data)].iloc[0]
    return f'{r.n:.{d}f} \\pm {r.n_se:.{d}f}'
V['NGnPeak'] = n('P+ (Pa)')
V['NGnLow'] = n('peak 0.05-4 Hz (Pa)')
V['NGnImp'] = n('I+ (Pa s)', d=1)
V['NGnPeakCorr'] = f"{dec[(dec.metric == 'P+ (Pa)') & (dec.data == 'B20 peak corrected')].n.iloc[0]:.2f}"
de = pd.read_csv(O / '34_amplitude_vs_distance' / 'decimation_effect.csv')
loss = 100 * (1 - de.ratio_P)
b14 = loss[de.id.str.contains('B14')]
oth = loss[~de.id.str.contains('B14')]
V['NGdecBfourteen'] = rng(b14.min(), b14.max())
V['NGdecOther'] = rng(np.floor(oth.min()), np.ceil(oth.max()))
V['NGdecMax'] = f'{loss.max():.0f}'
y = pd.read_csv(O / '34_amplitude_vs_distance' / 'yield_estimates.csv')
V['NGyieldP'] = f'{y.W_from_P_surface_t.median():.0f}'
V['NGyieldPrange'] = rng(y.W_from_P_surface_t.min(), y.W_from_P_surface_t.max())
V['NGyieldT'] = f'{y.W_from_tpos_surface_t.median():.0f}'
V['NGyieldTrange'] = rng(y.W_from_tpos_surface_t.min(), y.W_from_tpos_surface_t.max())
V['NGzRange'] = rng(np.floor(y.Z_from_P.min() / 10) * 10, np.ceil(y.Z_from_P.max() / 10) * 10)
V['NGyieldVsSilber'] = f'{100 * abs(y.W_from_P_surface_t.median() / 131 - 1):.0f}'

## Coupling (notebook 28)

In [4]:
cm = pd.read_csv(O / '28_air_ground_coupling' / 'coupling_metrics.csv')
c14 = cm[cm.station == 'B14']
V['NGcompBfourteen'] = f'{np.median(c14.compliance_nm_Pa):.0f}'
V['NGcompBfourteenRange'] = f'${c14.compliance_nm_Pa.max():.0f}$ to ${c14.compliance_nm_Pa.min():.0f}$'
V['NGcompBtwentynine'] = f"{cm[cm.station == 'B29'].compliance_nm_Pa.iloc[0]:.0f}"
V['NGcompBzerofive'] = f"{cm[cm.station == 'B05'].compliance_nm_Pa.iloc[0]:.0f}"
neg = cm[cm.compliance_nm_Pa < 0].compliance_nm_Pa.abs()
V['NGcompRange'] = rng(neg.min(), neg.max())
ok = cm[cm.station != 'B20']
V['NGkRms'] = rng(ok.k_rms_um_s_Pa.min(), ok.k_rms_um_s_Pa.max(), 1)
V['NGkPeak'] = rng(ok.k_peak_um_s_Pa.min(), ok.k_peak_um_s_Pa.max(), 1)
a2 = pd.read_csv(O / '28_air_ground_coupling' / 'artemis2_comparison.csv')
art = a2[a2.source.str.startswith('Artemis')].set_index('station')
ngx = a2[a2.source.str.startswith('New')].set_index('station')
V['NGcompAtwoBzerofive'] = f"{art.loc['B05', 'compliance_nm_Pa']:.1f}"
V['NGcompAtwoBtwentynine'] = f"{art.loc['B29', 'compliance_nm_Pa']:.1f}"
V['NGcompAgree'] = f"{100 * (art.compliance_nm_Pa / ngx.compliance_nm_Pa - 1).abs().max():.0f}"
lr = pd.read_csv(O / '28_air_ground_coupling' / 'layer_resonance.csv')
lr = lr[~lr.pair.str.startswith('B20')]
V['NGfzero'] = rng(lr.f0_hz.min(), lr.f0_hz.max(), 1)

## Seismic waves and velocity model (notebooks 26 and 38)

In [5]:
sk = pd.read_csv(O / '26_seismic_arrivals' / 'seismic_picks.csv').set_index('station').loc[['B14', 'B29', 'B05']]
aic = sk[['aic_2_12_s', 'aic_1_5_s']]
V['NGfirstArrival'] = rng(aic.min().min(), aic.max().max(), 1)
d = sk.distance_m / 1000
vmin, vmax = (d / aic.max(axis=1)).min(), (d / aic.min(axis=1)).max()
V['NGvFirst'] = rng(np.floor(vmin * 10) / 10, np.ceil(vmax * 10) / 10, 1)
V['NGlead'] = rng(sk.lead_over_airwave_s.min(), sk.lead_over_airwave_s.max())
je = pd.read_csv(O / '38_horizontal_components' / 'joint_ensemble.csv')
bestfits = je.sort_values('misfit').groupby('fit').head(1)
V['NGvsTop'] = rng(np.floor(bestfits.vs1_m_s.min() / 10) * 10, np.ceil(bestfits.vs1_m_s.max() / 10) * 10)
V['NGtopThick'] = rng(np.floor(bestfits.h1_m.min() / 5) * 5, np.ceil(bestfits.h1_m.max() / 5) * 5)
V['NGhalfDepth'] = rng(np.floor(bestfits.depth_to_halfspace_m.min() / 5) * 5, np.ceil(bestfits.depth_to_halfspace_m.max() / 5) * 5)
V['NGvsHalf'] = f'{np.round(bestfits.vs4_m_s.mean() / 10) * 10:.0f}'

## Chaparral (notebook 32)

In [6]:
cw = pd.read_csv(O / '32_chaparral_sensitivity' / 'chaparral_windows.csv')
ev = cw.groupby(['epoch', 'source']).S_V_per_Pa.median().reset_index()
def ep(e):
    g = ev[ev.epoch == e].S_V_per_Pa
    return rng(g.min(), g.max(), 2)
V['NGchapEarly'] = ep('Jan-4 Apr (500 sps)')
V['NGchapLate'] = ep('14 Jun-Sep (500 sps)')
V['NGchapExplosion'] = f"{cw[cw.source.str.startswith('explosion')].S_V_per_Pa.median():.3f}"
V['NGchapWindows'] = f'{len(cw)}'
V['NGchapLaunches'] = f"{cw[cw.source.str.startswith('launch')].source.nunique()}"

In [7]:
lines = ['% Generated by 42_paper_numbers.ipynb from the notebooks\' outputs; do not edit by hand.']
for k, v in V.items():
    lines.append(f'\\newcommand{{\\{k}}}{{{v}}}')
(cfg.PAPER_DIR / 'tables' / 'numbers.tex').write_text('\n'.join(lines) + '\n')
for k, v in V.items():
    print(f'{k:24s} {v}')

NGorigin                 01:00:02.4
NGoriginPrecise          01:00:02.37
NGoriginSpread           0.27
NGoriginRange            01:00:02.2--03.2
NGoriginFree             01:00:01.2
NGspeedFree              339
NGtemp                   26.4
NGrh                     87
NGpressure               1014
NGwind                   2.2
NGwindFrom               298
NGczero                  348.7
NGceff                   346.7--347.3
NGheadwind               1.5--2.0
NGpBtwenty               407
NGpFar                   264--342
NGtpos                   0.36--0.44
NGipos                   28--61
NGiposr                  580--1050
NGnegRatio               3--7
NGnPeak                  0.58 \pm 0.15
NGnLow                   1.23 \pm 0.09
NGnImp                   1.2 \pm 0.4
NGnPeakCorr              1.15
NGdecBfourteen           26--28
NGdecOther               1--4
NGdecMax                 28
NGyieldP                 145
NGyieldPrange            94--246
NGyieldT                 376
NGyieldTrange      